# Limpeza dos reviews
Preparar títulos e comentários para a base textual do RAG, mantendo os textos originais e os identificadores.

## 1. Carregar os dados
Os campos são lidos como texto e células vazias ficam como strings vazias.


In [ ]:
from pathlib import Path
from io import BytesIO
from IPython.display import display
import pandas as pd
import kagglehub
import zipfile

DATASET = "olistbr/brazilian-ecommerce/versions/2"
ARQUIVO = "olist_order_reviews_dataset.csv"

caminho = Path(
    kagglehub.dataset_download(DATASET, path=ARQUIVO)
)

opcoes_leitura = {
    "dtype": "string",
    "keep_default_na": False,
    "encoding": "utf-8",
}

if zipfile.is_zipfile(caminho):
    with zipfile.ZipFile(caminho) as zip_file:
        conteudo = zip_file.read(ARQUIVO)

    df_reviews_original = pd.read_csv(
        BytesIO(conteudo),
        **opcoes_leitura,
    )
else:
    df_reviews_original = pd.read_csv(
        caminho,
        **opcoes_leitura,
    )

print("Avaliações:", len(df_reviews_original))
print("Colunas:", df_reviews_original.columns.tolist())

display(df_reviews_original.head())

## 2. Verificar os textos disponíveis
Consideramos sem conteúdo os campos ausentes, vazios ou com apenas espaços.


In [ ]:
tem_comentario = (
    df_reviews_original["review_comment_message"]
    .fillna("")
    .str.strip()
    .ne("")
)

tem_titulo = (
    df_reviews_original["review_comment_title"]
    .fillna("")
    .str.strip()
    .ne("")
)

resumo_textos = pd.Series({
    "Título e comentário": (tem_titulo & tem_comentario).sum(),
    "Somente comentário": (~tem_titulo & tem_comentario).sum(),
    "Somente título": (tem_titulo & ~tem_comentario).sum(),
    "Sem título e sem comentário": (~tem_titulo & ~tem_comentario).sum(),
})

display(resumo_textos.to_frame("quantidade"))

## 3. Examinar avaliações somente com título
Há relatos sobre entrega e não recebimento, além de elogios genéricos. A amostra não garante que todos os títulos sejam úteis.

A conferência seguinte verifica se as quatro categorias somam as 99.224 avaliações carregadas.


In [ ]:
somente_titulo = tem_titulo & ~tem_comentario

exemplos = df_reviews_original.loc[
    somente_titulo,
    ["review_id", "order_id", "review_score", "review_comment_title"],
]

print("Avaliações somente com título:", len(exemplos))

display(
    exemplos.sample(
        n=min(15, len(exemplos)),
        random_state=42,
    )
)

In [ ]:
print("Total carregado:", len(df_reviews_original))
print("Soma das categorias:", resumo_textos.sum())
print(
    "Sem título e sem comentário:",
    (~tem_titulo & ~tem_comentario).sum(),
)

## 4. Selecionar avaliações com texto
Mantemos registros com título **ou** comentário preenchido. Isso evita perder relatos que aparecem apenas no título.

A seleção cria `df_reviews_preparado`, sem alterar `df_reviews_original`. Foram mantidas 42.687 avaliações e 56.537 ficaram fora do recorte textual.

In [ ]:
tem_texto = tem_comentario | tem_titulo

df_reviews_preparado = (
    df_reviews_original.loc[tem_texto].copy()
)

print("Avaliações selecionadas:", len(df_reviews_preparado))
print("Fora do recorte textual:", (~tem_texto).sum())

## 5. Padronizar espaços
Criamos uma coluna `_clean` para cada campo textual. Espaços repetidos, tabulações e quebras de linha viram um espaço; espaços nas bordas são removidos.

As colunas de texto original são mantidas. Não removemos acentos, números, pontuação, maiúsculas ou negações.


In [ ]:
for campo in ["review_comment_title", "review_comment_message"]:
    df_reviews_preparado[f"{campo}_clean"] = (
        df_reviews_preparado[campo]
        .fillna("")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

## 6. Conferir as alterações
Comparando cada texto original com sua versão limpa.


In [ ]:
for campo in ["review_comment_title", "review_comment_message"]:
    campo_limpo = f"{campo}_clean"

    alterados = (
        df_reviews_preparado[campo].fillna("")
        != df_reviews_preparado[campo_limpo]
    )

    print(f"{campo} — textos alterados:", alterados.sum())

    display(
        df_reviews_preparado.loc[
            alterados,
            [campo, campo_limpo],
        ].head(5)
    )

print("Avaliações mantidas:", len(df_reviews_preparado))

## 7. Verificar tags HTML
Procurando marcações como `<p>` e `<br>` nos textos limpos. Se nenhum caso encontrado, nenhuma conversão é aplicada.


In [ ]:
for campo in [
    "review_comment_title_clean",
    "review_comment_message_clean",
]:
    possivel_html = df_reviews_preparado[campo].str.contains(
        r"</?[A-Za-z][^>]*>",
        regex=True,
        na=False,
    )

    print(f"{campo} — possível HTML:", possivel_html.sum())

    display(
        df_reviews_preparado.loc[
            possivel_html,
            ["review_id", campo],
        ].head(5)
    )

## 8. Verificar entidades HTML
Procurando códigos como `&amp;` e `&nbsp;`. Se nenhum caso encontrado, nenhuma conversão é aplicada.


In [ ]:
for campo in [
    "review_comment_title_clean",
    "review_comment_message_clean",
]:
    possivel_entidade = df_reviews_preparado[campo].str.contains(
        r"&(?:[A-Za-z]+|#\d+|#x[0-9a-fA-F]+);",
        regex=True,
        na=False,
    )

    print(f"{campo} — possível entidade HTML:", possivel_entidade.sum())

    display(
        df_reviews_preparado.loc[
            possivel_entidade,
            ["review_id", campo],
        ].head(5)
    )

## 9. Conferir pedidos com mais de uma avaliação
Identificamos pedidos com várias avaliações e mostramos exemplos para comparar os relatos. Essa consulta não remove registros.


In [ ]:
avaliacoes_por_pedido = (
    df_reviews_preparado.groupby("order_id").size()
)

pedidos_com_multiplas = avaliacoes_por_pedido[
    avaliacoes_por_pedido > 1
].index

print("Pedidos com múltiplas avaliações:", len(pedidos_com_multiplas))

# Mostrar todas as avaliações de até 5 desses pedidos.
pedidos_exemplo = pedidos_com_multiplas[:5]

display(
    df_reviews_preparado.loc[
        df_reviews_preparado["order_id"].isin(pedidos_exemplo),
        [
            "order_id",
            "review_id",
            "review_score",
            "review_answer_timestamp",
            "review_comment_title_clean",
            "review_comment_message_clean",
        ],
    ].sort_values(["order_id", "review_answer_timestamp"])
)

Foram identificados 146 pedidos com múltiplas avaliações na base selecionada. Mantivemos os registros porque podem conter relatos diferentes ou complementares.

A EDA usa a avaliação mais recente para indicadores por pedido. Essa regra não foi aplicada à base textual. Quantidade de avaliações e quantidade de pedidos são medidas diferentes.


## 10. Verificar linhas completamente repetidas
A comparação usa todas as colunas. Se nenhum caso encontrado, nenhuma conversão é aplicada.


In [ ]:
duplicadas = df_reviews_preparado.duplicated()

print("Linhas completamente duplicadas:", duplicadas.sum())

display(
    df_reviews_preparado.loc[duplicadas].head(5)
)

## 11. Conferir identificadores ausentes
Os campos `review_id` e `order_id` permitem localizar a origem dos textos. Se nenhum caso encontrado, nenhuma alteração é aplicada.. Isso ainda não significa que cada identificador seja único.


In [ ]:
for campo in ["review_id", "order_id"]:
    ausentes = (
        df_reviews_preparado[campo]
        .fillna("")
        .str.strip()
        .eq("")
    )

    print(f"{campo} — ausentes:", ausentes.sum())

    display(
        df_reviews_preparado.loc[
            ausentes,
            ["review_id", "order_id"],
        ].head(5)
    )

## 12. Conferir repetição de review_id
Foram encontrados 307 IDs repetidos, envolvendo 624 linhas. Por isso, `review_id` não deve ser usado sozinho como chave única desta base.

In [ ]:
ids_repetidos = df_reviews_preparado["review_id"].duplicated(
    keep=False
)

print(
    "review_ids distintos que se repetem:",
    df_reviews_preparado.loc[
        ids_repetidos, "review_id"
    ].nunique(),
)

print("Linhas envolvidas:", ids_repetidos.sum())

display(
    df_reviews_preparado.loc[
        ids_repetidos,
        [
            "review_id",
            "order_id",
            "review_comment_title_clean",
            "review_comment_message_clean",
        ],
    ].sort_values(["review_id", "order_id"]).head(10)
)

## 13. Conferir a chave composta
O par `review_id` + `order_id` não se repete na base selecionada. Os dois campos devem acompanhar a entrega para identificar cada registro.


In [ ]:
pares_repetidos = df_reviews_preparado.duplicated(
    subset=["review_id", "order_id"],
    keep=False,
)

print("Linhas com par review_id/order_id repetido:", pares_repetidos.sum())

display(
    df_reviews_preparado.loc[
        pares_repetidos,
        [
            "review_id",
            "order_id",
            "review_comment_title_clean",
            "review_comment_message_clean",
        ],
    ].sort_values(["review_id", "order_id"]).head(10)
)

## 14. Validar as notas
Veficar se as notas podem ser interpretadas como números de 1 a 5. Se nenhum caso encontrado, nenhuma conversão é aplicada.


In [ ]:
notas = pd.to_numeric(
    df_reviews_preparado["review_score"],
    errors="coerce",
)

notas_invalidas = ~notas.isin([1, 2, 3, 4, 5])

print("Notas ausentes ou inválidas:", notas_invalidas.sum())

display(
    df_reviews_preparado.loc[
        notas_invalidas,
        ["review_id", "order_id", "review_score"],
    ].head(10)
)

## 15. Converter as notas
A coluna `review_score` passa de texto para inteiro no DataFrame. Os valores das notas são mantidos.


In [ ]:
df_reviews_preparado["review_score"] = notas.astype("Int64")

print("Tipo da coluna:", df_reviews_preparado["review_score"].dtype)

## 16. Validar as datas
Verificar se as datas preenchidas seguem o formato esperado e distinguimos ausências de falhas de conversão.


In [ ]:
datas_convertidas = {}

for campo in ["review_creation_date", "review_answer_timestamp"]:
    datas_convertidas[campo] = pd.to_datetime(
        df_reviews_preparado[campo],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
    )

    ausentes = (
        df_reviews_preparado[campo]
        .fillna("")
        .str.strip()
        .eq("")
    )

    invalidas = ~ausentes & datas_convertidas[campo].isna()

    print(f"{campo} — ausentes:", ausentes.sum())
    print(f"{campo} — preenchidas não convertidas:", invalidas.sum())

    display(
        df_reviews_preparado.loc[
            invalidas,
            ["review_id", campo],
        ].head(5)
    )

## 17. Aplicar a conversão das datas
As colunas passam para datetime no DataFrame, mantendo data e horário. Essa etapa valida e padroniza a representação; o CSV não preservará o tipo datetime.


In [ ]:
for campo, datas in datas_convertidas.items():
    df_reviews_preparado[campo] = datas

display(
    df_reviews_preparado[
        ["review_creation_date", "review_answer_timestamp"]
    ].dtypes.to_frame("tipo")
)

## 18. Conferir o recorte após a limpeza
Cada registro deve continuar com pelo menos um dos campos limpos preenchido. Título vazio ou comentário vazio isoladamente é permitido.

Resultado salvo: 42.687 avaliações e nenhum registro com os dois campos limpos vazios.


In [ ]:
titulo_vazio = (
    df_reviews_preparado["review_comment_title_clean"]
    .fillna("")
    .str.strip()
    .eq("")
)

comentario_vazio = (
    df_reviews_preparado["review_comment_message_clean"]
    .fillna("")
    .str.strip()
    .eq("")
)

sem_texto = titulo_vazio & comentario_vazio

print("Avaliações preparadas:", len(df_reviews_preparado))
print("Sem título e sem comentário após limpeza:", sem_texto.sum())

## 19. Exportar o CSV
Salvamos as 42.687 avaliações com os campos originais e as duas colunas de texto limpo, sem juntar título e comentário.

O caminho `../data/processed` pressupõe execução na pasta `notebooks`. Executar novamente substitui o CSV de mesmo nome. Datas são escritas como `YYYY-MM-DD HH:MM:SS`; os tipos devem ser definidos na leitura seguinte.


In [ ]:
pasta_saida = Path("../data/processed")
pasta_saida.mkdir(parents=True, exist_ok=True)

arquivo_saida = pasta_saida / "olist_reviews_preparados.csv"

df_reviews_preparado.to_csv(
    arquivo_saida,
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d %H:%M:%S",
)

print("CSV salvo em:", arquivo_saida.resolve())

## 20. Salvar o resumo
O arquivo Markdown reúne as regras, os resultados e as limitações desta preparação. As contagens no texto abaixo foram conferidas com as saídas salvas desta execução; se mudar a fonte ou o recorte, é necessário a atualização do resumo.
O caminho `../docs` pressupõe execução na pasta `notebooks`. A entrega é o CSV preparado, este notebook e o resumo.


In [ ]:
resumo = """# Preparação das avaliações — Tech Challenge Fase 4

## Fonte
Dataset: olistbr/brazilian-ecommerce, versão 2.
Arquivo: olist_order_reviews_dataset.csv.

## Seleção dos dados
- Base original: 99.224 avaliações.
- Título e comentário: 9.829.
- Somente comentário: 31.121.
- Somente título: 1.737.
- Sem título e sem comentário: 56.537.
- Base selecionada: 42.687 avaliações.

Foram mantidas avaliações com título ou comentário preenchido.
A amostra de títulos mostrou evidências úteis, como relatos sobre
prazo de entrega e não recebimento.
Registros sem texto nos dois campos ficaram fora do recorte textual.
Nenhum conteúdo foi inventado para preencher ausências.

## Limpeza dos textos
- Espaços repetidos, tabulações e quebras de linha foram
  padronizados para um único espaço.
- Espaços nas bordas foram removidos.
- Títulos alterados: 1.997.
- Comentários alterados: 11.475.
- Os textos originais foram preservados em colunas separadas.
- Acentos, números, pontuação, maiúsculas e negações foram mantidos.
- Não foram aplicados stemming, lematização ou remoção de stop words.
- Não foram encontradas tags ou entidades HTML nos títulos
  e comentários pelos padrões verificados.

## Avaliações e identificadores
- Foram identificados 146 pedidos com múltiplas avaliações
  na base textual selecionada.
- Avaliações distintas do mesmo pedido foram preservadas,
  pois podem conter informações complementares.
- Não foram encontradas linhas completamente duplicadas.
- review_id e order_id não apresentaram valores ausentes.
- 307 review_ids se repetem, envolvendo 624 linhas.
- O par review_id + order_id é único na base selecionada.
- Os dois identificadores devem ser preservados na próxima etapa.

## Notas e datas
- Todas as notas foram validadas no intervalo de 1 a 5.
- review_score foi convertido para inteiro no DataFrame.
- review_creation_date e review_answer_timestamp foram
  validados e convertidos para datetime.
- Não foram encontrados valores ausentes ou inválidos nesses campos.
- O CSV não preserva os tipos do pandas. Datas foram exportadas
  no formato YYYY-MM-DD HH:MM:SS.

## Validação final
- Avaliações preparadas: 42.687.
- Registros com título e comentário limpos simultaneamente vazios: 0.

## Entrega e uso
- Arquivo: olist_reviews_preparados.csv.
- Título preparado: review_comment_title_clean.
- Comentário preparado: review_comment_message_clean.
- Título e comentário permanecem separados.
- Organização dos documentos, embeddings, armazenamento
  e indexação pertencem à próxima etapa.

## Limitações
- O corpus representa apenas avaliações com conteúdo textual.
- A quantidade de avaliações não equivale à quantidade de pedidos.
- Textos genéricos, como elogios curtos, não foram excluídos.
- A limpeza de formatação não garante que todo texto contenha
  evidência útil; por exemplo, 'Não quero avaliar' requer atenção.
"""

path_docs = Path("../docs")
path_docs.mkdir(parents=True, exist_ok=True)

arquivo_resumo = path_docs / "resumo_preparacao.md"
arquivo_resumo.write_text(resumo, encoding="utf-8")

print("Resumo salvo em:", arquivo_resumo.resolve())